# Databricks로 첫 번째 에이전트 시스템 구축하기

<img src="https://github.com/databricks-demos/dbdemos-resources/blob/main/images/product/llm-tools-functions/ai-agent-functions.png?raw=true" style="float: right" width="700px">

LLM은 일반적인 지식에 답변하는 데 강력하지만, 여러분의 비즈니스 정보는 가지고 있지 않습니다.

Databricks를 사용하면 LLM이 호출할 수 있는 커스텀 도구를 쉽게 개발하여 추론 능력을 갖춘 완전한 에이전트를 만들 수 있습니다.

### 간단한 UC 도구 구축하기
이 노트북에서는 다음을 Unity Catalog에 등록합니다:
- **SQL 함수**: 고객 정보, 주문 및 구독 정보에 접근하는 쿼리를 생성합니다.
- **간단한 Python 함수**: 언어 모델의 일반적인 한계를 극복하기 위해 수학 연산을 수행하는 Python 함수를 생성하고 등록합니다.

<!-- Collect usage data (view). Remove it to disable collection or disable tracker during installation. View README for more details.  -->
<img width="1px" src="https://ppxrzfxige.execute-api.us-west-2.amazonaws.com/v1/analytics?category=data-science&org_id=601859075515029&notebook=%2F01-create-tools%2F01_create_first_billing_agent&demo_name=ai-agent&event=VIEW&path=%2F_dbdemos%2Fdata-science%2Fai-agent%2F01-create-tools%2F01_create_first_billing_agent&version=1">

참고: 이 데모는 서버리스 컴퓨팅으로 실행하세요 - DBR 17.1에서도 테스트되었습니다

In [0]:
%pip install -U -qqqq mlflow>=3.1.4 databricks-langchain pydantic databricks-agents unitycatalog-langchain[databricks] databricks-feature-engineering==0.12.1 protobuf<5  cryptography<43 
# Restart to load the packages into the Python environment
dbutils.library.restartPython()

In [0]:
%run ../_resources/01-setup

에이전트에서 사용할 합성 데이터를 생성합니다.

## 1. 이메일을 기반으로 고객 정보 가져오기

이메일을 기반으로 고객 상세 정보를 검색하는 함수를 추가하겠습니다.

*참고: 이러한 포인트 쿼리에 낮은 지연 시간을 제공하기 위해 실시간 Postgres 테이블을 사용할 수 있습니다. 하지만 이 데모를 간단하게 유지하기 위해 SQL 엔드포인트를 사용하겠습니다. SQL 엔드포인트를 사용하는 것은 에이전트에 분석 기능을 제공하는 매우 좋은 방법이기도 합니다!*

In [0]:
%sql
CREATE OR REPLACE FUNCTION get_customer_by_email(email_input STRING COMMENT '고객 정보 추출에 사용된 이메일')
RETURNS TABLE (
    customer_id BIGINT,
    first_name STRING,
    last_name STRING,
    email STRING,
    phone STRING,
    address STRING,
    city STRING,
    state STRING,
    zip_code STRING,
    customer_segment STRING,
    registration_date DATE,
    customer_status STRING,
    loyalty_tier STRING,
    tenure_years DOUBLE,
    churn_risk_score BIGINT,
    customer_value_score BIGINT
)
COMMENT '입력한 이메일 주소와 일치하는 고객 정보를 조회합니다. 고객 ID, 이름, 성 등을 포함합니다.'
RETURN (
    SELECT * FROM customers
    WHERE email = email_input
    LIMIT 1
);


In [0]:
%sql SELECT * FROM get_customer_by_email('john21@example.net');

## 2. 모든 청구 정보 검색하기
고객의 모든 주문 및 구독 정보를 가져오는 함수를 추가하겠습니다. 이 함수는 고객 ID를 입력으로 받아 모든 과거 청구 정보와 현재 구독을 필터링하기 위한 집계를 반환합니다.

In [0]:
%sql
CREATE OR REPLACE FUNCTION get_customer_billing_and_subscriptions(customer_id_input BIGINT COMMENT '주문, 결제 및 구독을 검색하는 데 사용되는 고객 ID')
RETURNS TABLE (
    customer_id BIGINT,
    subscription_id BIGINT,
    service_type STRING,
    plan_name STRING,
    plan_tier STRING,
    monthly_charge BIGINT,
    start_date DATE,
    contract_length_months BIGINT,
    status STRING,
    autopay_enabled BOOLEAN,
    total_billed DOUBLE,
    total_paid DOUBLE,
    total_late_payments BIGINT,
    total_late_fees DOUBLE,
    latest_payment_status STRING
)
COMMENT '고객의 구독 정보와 결제 상세 내역을 조회합니다.'
RETURN (
    SELECT
        s.customer_id, s.subscription_id, s.service_type, s.plan_name, s.plan_tier,
        s.monthly_charge, s.start_date, s.contract_length_months, s.status, s.autopay_enabled,
        COALESCE(b.total_billed, 0), COALESCE(b.total_paid, 0),
        COALESCE(b.total_late_payments, 0), COALESCE(b.total_late_fees, 0),
        COALESCE(b.latest_payment_status, 'N/A')
    FROM subscriptions s
    LEFT JOIN (
        SELECT
            subscription_id, customer_id,
            SUM(total_amount) AS total_billed,
            SUM(payment_amount) AS total_paid,
            COUNT_IF(payment_date > due_date OR payment_status = 'Late') AS total_late_payments,
            SUM(CASE WHEN payment_date > due_date OR payment_status = 'Late' THEN total_amount - payment_amount ELSE 0 END) AS total_late_fees,
            MAX(payment_status) AS latest_payment_status
        FROM billing
        WHERE customer_id = customer_id_input
        GROUP BY subscription_id, customer_id
    ) b ON s.subscription_id = b.subscription_id
    WHERE s.customer_id = customer_id_input
);

In [0]:
%sql
SELECT *
FROM get_customer_billing_and_subscriptions(
  (SELECT customer_id FROM get_customer_by_email('john21@example.net'))
);


## 3. LLM에 수학 계산을 위한 Python 함수 제공하기
LLM은 일반적으로 고급 수학 계산에 어려움을 겪습니다. Python을 직접 사용하여 LLM이 수학 표현식을 계산할 수 있도록 도구를 추가하겠습니다.

Databricks는 안전한 샌드박스 Python 함수를 쉽게 실행할 수 있습니다:

In [0]:
# -----------------------
# 도구 2: 수학 표현식 계산하기
# -----------------------
def calculate_math_expression(expression: str) -> float:
    """
    기본 수학 표현식을 안전하게 평가합니다.

    Args:
        expression (str): 수학 표현식 (예: "sqrt(2 + 3 * (4 - 1))", Python math 함수 사용).

    Returns:
        float: 평가된 표현식의 결과.
    """
    import math
    allowed_names = {k: v for k, v in math.__dict__.items() if not k.startswith("__")}
    allowed_names.update({"abs": abs, "round": round})

    try:
        result = eval(expression, {"__builtins__": None}, allowed_names)
        return float(result)
    except Exception as e:
        raise ValueError(f"잘못된 표현식: {expression}. 오류: {str(e)}")

print(calculate_math_expression("6 + 3 * (13 - 1)"))

In [0]:
from unitycatalog.ai.core.databricks import DatabricksFunctionClient

client = DatabricksFunctionClient()

# UC에 도구를 배포하며, 도구의 docstring 및 타입 힌트를 기반으로 UC의 메타데이터를 자동으로 설정합니다
python_tool_uc_info = client.create_python_function(func=calculate_math_expression, catalog=catalog, schema=dbName, replace=True)

# 도구는 UC의 함수로 `{catalog}.{schema}.{func}` 형태로 배포됩니다. 여기서 {func}는 함수 이름입니다
# 배포된 Unity Catalog 함수 이름 출력
print(f"배포된 Unity Catalog 함수 이름: {python_tool_uc_info.full_name}")
# 생성된 함수로 이동하는 HTML 링크 생성
displayHTML(f'<a href="/explore/data/functions/{catalog}/{dbName}/calculate_math_expression" target="_blank">Unity Catalog에서 등록된 함수 보기</a>')

###### 참고: System.ai.python_exec에도 LLM이 샌드박스 환경에서 생성된 코드를 실행할 수 있게 해주는 함수가 등록되어 있습니다:

```

%sql
SELECT system.ai.python_exec("""
from datetime import datetime
print(datetime.now().strftime('%Y-%m-%d'))
""") as current_date

```

## 4: AI Playground로 이동하여 이러한 함수를 사용하고 첫 번째 에이전트를 구성하는 방법을 살펴보세요!

[Playground](/ml/playground)를 열고 우리가 생성한 도구를 선택하여 에이전트를 테스트하세요!

<div style="float: right; width: 70%;">
  <img 
    src="https://raw.githubusercontent.com/databricks-demos/dbdemos-resources/refs/heads/main/images/\
cross_demo_assets/AI_Agent_GIFs/AI_agent_function_selection.gif" 
    alt="Function Selection" 
    width="100%"
  >
</div>

### 위치 안내

함수는 Unity Catalog에서 다음과 같은 구조로 구성됩니다:

#### 예시 경로:
`my_catalog.my_schema.my_awesome_function`

💡 참고: 예시 이름을 실제 카탈로그 및 스키마 이름으로 바꾸세요.

## 다음 단계: 평가

이제 에이전트가 준비되었고 도구를 활용하여 질문에 적절히 답변할 수 있습니다.

하지만 어떻게 제대로 작동하는지, 그리고 더 중요하게는 향후 질문과 수정 사항에 대해서도 잘 작동할 것인지 확인할 수 있을까요?

이를 위해 평가 데이터셋을 구축하고 MLFlow를 활용하여 에이전트를 자동으로 분석해야 합니다!
[02-agent-eval/02.1_agent_evaluation]($../02-agent-eval/02.1_agent_evaluation) 노트북을 열어 Langchain을 사용하여 에이전트를 배포하고 첫 번째 평가를 실행하는 방법을 확인하세요!